Use wb_command to resample NSD fsaverage NCSNR to fsLR32k space

This notebook resamples NSD subject specific NCSNR estimates from fsaverage to fsLR32k space. It can be divided into three parts:
1. Visualize subject specific NCSNR on fsaverage flatmap
2. Resample fsaverage map to fsLR32k space
3. Visualize subject specific ROI classes on fsLR32k flatmap

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
import sys
sys.path.append(os.getenv('PYTHONPATH'))
import subprocess
import nibabel as nib
import numpy as np
import hcp_utils as hcp
import matplotlib.pyplot as plt
from nilearn import plotting, datasets
import cortex
import pandas as pd
import seaborn as sns
cortex.download_subject('fsaverage')

from src.utils.transforms import SelectROIs

In [ ]:
#housekeeping
datasets_root = os.path.join(os.getenv("DATASETS_ROOT", "/default/path/to/datasets")) #use default if DATASETS_ROOT env variable is not set.
dataset_root = os.path.join(datasets_root, "NaturalScenesDataset")
meta_dataset_root = os.path.join(datasets_root, "MOSAIC")
project_root = os.getenv("PROJECT_ROOT", "/default/path/to/datasets")
working_path = os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output")
subjects = [f"sub-{x:02d}" for x in range(1,9)]

In [ ]:
vertex_info = hcp.vertex_info
rois = [f'GlasserGroup_{x}' for x in range(1,23)]
ROIselection = SelectROIs(selected_rois=rois)
cols = ['subjectID', 'mosaic', 'original', 'GlasserGroup']
results = {col: [] for col in cols}
for subject in subjects:
    #load the mosaic nsd ncsnr
    lh_ncsnr_mosaic = np.load(os.path.join(working_path, "ncsnr_mosaic", subject, f"lh.ncsnr_mosaic.npy"))
    rh_ncsnr_mosaic = np.load(os.path.join(working_path, "ncsnr_mosaic", subject, f"rh.ncsnr_mosaic.npy"))
    data_mosaic = np.hstack((lh_ncsnr_mosaic[vertex_info.grayl],rh_ncsnr_mosaic[vertex_info.grayr]))

    #load the original (resampled) nsd ncsnr
    lh_ncsnr_original = np.load(os.path.join(working_path, "ncsnr_original", subject, f"lh.ncsnr_fsLR32k_space_resampled.npy"))
    rh_ncsnr_original = np.load(os.path.join(working_path, "ncsnr_original", subject, f"rh.ncsnr_fsLR32k_space_resampled.npy"))
    data_original = np.hstack((lh_ncsnr_original[vertex_info.grayl],rh_ncsnr_original[vertex_info.grayr]))

    for roi in rois:
        # Get the arrays for this ROI
        original_vals = data_original[ROIselection.group_to_index[roi]]
        mosaic_vals = data_mosaic[ROIselection.group_to_index[roi]]
        
        # Append each individual value instead of the whole array
        for orig, mos in zip(original_vals, mosaic_vals):
            results['subjectID'].append(subject)
            results['original'].append(orig)
            results['mosaic'].append(mos)
            results['GlasserGroup'].append(roi)




In [ ]:
df = pd.DataFrame(results)
for subject in subjects:
    df_subject = df[df['subjectID'] == subject]
    #df_subject.dropna(subset=['mosaic', 'original'])
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_subject, x='original', y='mosaic', hue='GlasserGroup', s=10, alpha=0.7)

    # Add diagonal red line
    min_val = min(df_subject['original'].min(), df_subject['mosaic'].min())
    max_val = max(df_subject['original'].max(), df_subject['mosaic'].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r-', linewidth=2, label='y=x')

    plt.xlabel('Original', fontsize=12)
    plt.ylabel('Mosaic', fontsize=12)
    plt.title(f'{subject} Mosaic vs Original by Glasser Group', fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    

In [ ]:
for subject in subjects:
    #load the mosaic nsd ncsnr
    lh_ncsnr_mosaic = np.load(os.path.join(working_path, "ncsnr_mosaic", subject, f"lh.ncsnr_mosaic.npy"))
    rh_ncsnr_mosaic = np.load(os.path.join(working_path, "ncsnr_mosaic", subject, f"rh.ncsnr_mosaic.npy"))
    data_mosaic = np.hstack((lh_ncsnr_mosaic[vertex_info.grayl],rh_ncsnr_mosaic[vertex_info.grayr]))

    #load the original (resampled) nsd ncsnr in fsaverage space
    lh_ncsnr_original = np.squeeze(nib.load(os.path.join(working_path, "ncsnr_original", subject, f"lh.ncsnr.mgh")).get_fdata())
    rh_ncsnr_original = np.squeeze(nib.load(os.path.join(working_path, "ncsnr_original", subject, f"rh.ncsnr.mgh")).get_fdata())
    data_original = np.hstack((lh_ncsnr_original,rh_ncsnr_original))

    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Overlaid histograms (density/percentage)
    axes[0].hist(data_mosaic, bins=60, alpha=0.6, label='MOSAIC', density=True, color='blue')
    axes[0].hist(data_original, bins=60, alpha=0.6, label='Original fsaverage', density=True, color='orange')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Distribution Comparison (Normalized)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot 2: Cumulative percentage above threshold
    thresholds = np.linspace(min(np.nanmin(data_mosaic), np.nanmin(data_original)), 
                            max(np.nanmax(data_mosaic), np.nanmax(data_original)), 100)

    percent_above_1 = [100 * np.nansum(data_mosaic >= t) / len(data_mosaic) for t in thresholds]
    percent_above_2 = [100 * np.nansum(data_original >= t) / len(data_original) for t in thresholds]

    axes[1].plot(thresholds, percent_above_1, label='MOSAIC', linewidth=2, color='blue')
    axes[1].plot(thresholds, percent_above_2, label='Original fsaverage', linewidth=2, color='orange')
    axes[1].set_xlabel('Threshold Value')
    axes[1].set_ylabel('Percent of Elements Above Threshold (%)')
    axes[1].set_title('Cumulative Percentage Above Threshold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 105])
    plt.suptitle(f'{subject} Comparison between Original and MOSAIC NSD')
    plt.tight_layout()
    plt.show()

In [ ]:
save_flag = False
# Create figure with 2 rows, 4 columns
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()  # Flatten to make indexing easier
lw = 3
for idx, subject in enumerate(subjects):
    #load the mosaic nsd ncsnr
    lh_ncsnr_mosaic = np.load(os.path.join(working_path, "ncsnr_mosaic", subject, f"lh.ncsnr_mosaic.npy"))
    rh_ncsnr_mosaic = np.load(os.path.join(working_path, "ncsnr_mosaic", subject, f"rh.ncsnr_mosaic.npy"))
    data_mosaic = np.hstack((lh_ncsnr_mosaic[vertex_info.grayl],rh_ncsnr_mosaic[vertex_info.grayr]))

    #load the original (resampled) nsd ncsnr in fsaverage space
    lh_ncsnr_original = np.squeeze(nib.load(os.path.join(working_path, "ncsnr_original", subject, f"lh.ncsnr.mgh")).get_fdata())
    rh_ncsnr_original = np.squeeze(nib.load(os.path.join(working_path, "ncsnr_original", subject, f"rh.ncsnr.mgh")).get_fdata())
    data_original = np.hstack((lh_ncsnr_original,rh_ncsnr_original))

    # Plot cumulative percentage above threshold
    thresholds = np.linspace(min(np.nanmin(data_mosaic), np.nanmin(data_original)), 
                            max(np.nanmax(data_mosaic), np.nanmax(data_original)), 100)

    percent_above_1 = [100 * np.nansum(data_mosaic >= t) / len(data_mosaic) for t in thresholds]
    percent_above_2 = [100 * np.nansum(data_original >= t) / len(data_original) for t in thresholds]

    axes[idx].plot(thresholds, percent_above_1, label='MOSAIC', linewidth=lw, color='blue')
    axes[idx].plot(thresholds, percent_above_2, label='Original fsaverage', linewidth=lw, color='orange')
    axes[idx].set_xlabel('Threshold Value')
    axes[idx].set_ylabel('Percent Above Threshold (%)')
    axes[idx].set_title(f'{subject}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_yscale('log')
    axes[idx].set_ylim([0.1, 105])

plt.suptitle('Cumulative Percentage Above Threshold: Original vs MOSAIC NSD', fontsize=16)
plt.tight_layout()
if save_flag:
    plt.savefig(os.path.join(working_path, "plots", "compare_ncsnr_mosaic_fsaverage.png"), dpi=300)
    plt.savefig(os.path.join(working_path, "plots", "compare_ncsnr_mosaic_fsaverage.svg"))
plt.show()

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Overlaid histograms (density/percentage)
axes[0].hist(data_mosaic, bins=60, alpha=0.6, label='MOSAIC', density=True, color='blue')
axes[0].hist(data_original, bins=60, alpha=0.6, label='Original', density=True, color='orange')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution Comparison (Normalized)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative percentage above threshold
thresholds = np.linspace(min(np.nanmin(data_mosaic), np.nanmin(data_original)), 
                         max(np.nanmax(data_mosaic), np.nanmax(data_original)), 100)

percent_above_1 = [100 * np.nansum(data_mosaic >= t) / len(data_mosaic) for t in thresholds]
percent_above_2 = [100 * np.nansum(data_original >= t) / len(data_original) for t in thresholds]

axes[1].plot(thresholds, percent_above_1, label='MOSAIC', linewidth=2, color='blue')
axes[1].plot(thresholds, percent_above_2, label='Original', linewidth=2, color='orange')
axes[1].set_xlabel('Threshold Value')
axes[1].set_ylabel('Percent of Elements Above Threshold (%)')
axes[1].set_title('Cumulative Percentage Above Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 105])
plt.suptitle(f'{subject} Comparison between Original and MOSAIC NSD')
plt.tight_layout()
plt.show()

# Optional: Print some statistics
print(f"MOSAIC: mean={np.nanmean(data_mosaic):.3f}, std={np.nanstd(data_mosaic):.3f}, size={len(data_mosaic)}")
print(f"Original: mean={np.nanmean(data_original):.3f}, std={np.nanstd(data_original):.3f}, size={len(data_original)}")
print(f"\nPercent above median of MOSAIC:")
print(f"  MOSAIC: {100 * np.nansum(data_mosaic > np.nanmedian(data_mosaic)) / len(data_mosaic):.1f}%")
print(f"  Original: {100 * np.nansum(data_original > np.nanmedian(data_original)) / len(data_original):.1f}%")

In [ ]:
np.nansum(data_original == 0) / len(data_original)

In [ ]:
np.nansum(data_mosaic == 0) / len(data_mosaic)

In [ ]:
np.isnan(data_mosaic).sum()